In [1]:
print("all ok")


all ok


Tool calling means an LLM can decide to use an external function/tool when it needs information or wants to perform an action, instead of answering only from its own knowledge.

Tool calling is the ability of an LLM to select and invoke external tools or functions to get information or perform actions beyond its built-in knowledge.

## these are some of the example 
### but the concept is we can convert any of the funcationality into the tool

Web search
Calculator
Database query
API calls
retriever
Email sending
Weather API
Stock-price API
File operations

User: "What is the weather in Bangalore today?"

        ↓

LLM decides: "I need current weather data."

        ↓

Calls Weather Tool

        ↓

Tool returns: 28°C, cloudy

        ↓

LLM gives final answer

Node = graph decides when to execute the function.
Tool = LLM decides when to execute the function.

User
 ↓
LLM
 ↓
Does it need a tool?
 ↓
Yes
 ↓
Tool(function) Call
 ↓
Tool Result
 ↓
LLM
 ↓
Final Answer

MCP Server
   │
   ├── Tool 1
   ├── Tool 2
   └── Tool 3
        ↓
langchain-mcp-adapters
        ↓
LangChain BaseTools
        ↓
LangGraph Agent

Now if the user asks: "What is 20 + 30?"

the LLM may produce a tool request like:

Tool: add
Arguments:
a = 20
b = 30

Then the tool executes: 50

and the LLM can use that result to answer.

The LLM does not execute the function itself. It decides which tool(function) to call and with what arguments; your application executes the tool and returns the result to the LLM.

LLM = decision maker
Tool = actual executor(acutal funcationality)

In [1]:
from dotenv import load_dotenv
import os
load_dotenv()

print("Setup loaded.")
print("GROQ_API_KEY available:", bool(os.getenv("GROQ_API_KEY")))
print("OPENAI_API_KEY available:", bool(os.getenv("OPENAI_API_KEY")))
print("TAVILY_API_KEY available:", bool(os.getenv("TAVILY_API_KEY")))

Setup loaded.
GROQ_API_KEY available: True
OPENAI_API_KEY available: True
TAVILY_API_KEY available: True


## 1. `@tool` decorator

In [2]:
from langchain_core.tools import tool

In [3]:
@tool
def add_basic(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [4]:
print("Name:", add_basic.name)

Name: add_basic


In [5]:
print("Description:", add_basic.description)

Description: Add two numbers.


In [6]:
print("Args:", add_basic.args)

Args: {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [7]:
add_basic.invoke({"a": 20, "b": 30})

50

In [8]:
result = add_basic.invoke({"a": 20, "b": 30})
print("Execution result:", result)

Execution result: 50


## 2. `@tool("custom_name")`

In [9]:
@tool("calculator")
def add_with_custom_name(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

In [10]:
print("Tool name:", add_with_custom_name.name)
print("Execution result:", add_with_custom_name.invoke({"a": 10, "b": 15}))

Tool name: calculator
Execution result: 25


In [11]:
@tool(
    "multiply_numbers",
    description="Multiply two integers and return the result.",
    return_direct=False,
)
def multiply_with_options(a, b):
    return a * b

In [12]:
print("Execution result:", multiply_with_options.invoke({"a": "sunny", "b": 7}))

Execution result: sunnysunnysunnysunnysunnysunnysunny


In [13]:
print("Name:", multiply_with_options.name)
print("Description:", multiply_with_options.description)
print("return_direct:", multiply_with_options.return_direct)
print("Execution result:", multiply_with_options.invoke({"a": 6, "b": 7}))

Name: multiply_numbers
Description: Multiply two integers and return the result.
return_direct: False
Execution result: 42


## 4. `@tool` with Pydantic

In [14]:
from pydantic import BaseModel, Field, ValidationError

In [15]:
class CalculatorInputTest(BaseModel):
    a: int = Field(description="First integer")
    b: int = Field(description="Second integer")

In [16]:
@tool(args_schema=CalculatorInputTest)
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [17]:
print("Schema:", multiply.args_schema.model_json_schema())

Schema: {'properties': {'a': {'description': 'First integer', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'Second integer', 'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'CalculatorInputTest', 'type': 'object'}


In [18]:
print("Execution result:", multiply.invoke({"a": 8, "b": 9}))

Execution result: 72


In [19]:
multiply.invoke({"a": "sunny", "b": 9})

ValidationError: 1 validation error for CalculatorInputTest
a
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='sunny', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing

## 5. `@tool(parse_docstring=True)`

In [20]:
@tool(parse_docstring=True)
def search_with_docstring(query: str, limit: int) -> str:
    """Search documents.

    Args:
        query: Search query entered by the user.
        limit: Maximum number of results.
    """
    return f"Searching for '{query}' with limit={limit}"

In [21]:
print("Args schema:")
print(search_with_docstring.args_schema.model_json_schema())

Args schema:
{'description': 'Search documents.', 'properties': {'query': {'description': 'Search query entered by the user.', 'title': 'Query', 'type': 'string'}, 'limit': {'description': 'Maximum number of results.', 'title': 'Limit', 'type': 'integer'}}, 'required': ['query', 'limit'], 'title': 'search_with_docstring', 'type': 'object'}


In [22]:
print("Execution result:", search_with_docstring.invoke({"query": "LangGraph memory","limit": 3,}))


Execution result: Searching for 'LangGraph memory' with limit=3


## 6. Async function with `@tool`

In [23]:
import asyncio

In [24]:
@tool
async def get_data_async(url: str) -> str:
    """Fetch data asynchronously."""
    await asyncio.sleep(0.1)
    return f"Data from {url}"

In [25]:
result = await get_data_async.ainvoke({"url": "https://example.com"})
print("Async execution result:", result)

Async execution result: Data from https://example.com


## 7. Tool with `ToolRuntime` 

<!-- from typing_extensions import TypedDict
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.prebuilt import ToolNode, ToolRuntime
from langchain_core.messages import AIMessage

class RuntimeState(MessagesState):
    question: str

@tool
def read_question_from_runtime(runtime: ToolRuntime) -> str:
    """Read the current question from LangGraph state."""
    return runtime.state["question"]

def create_demo_tool_call(state: RuntimeState):
    return {
        "messages": [
            AIMessage(
                content="",
                tool_calls=[{
                    "name": "read_question_from_runtime",
                    "args": {},
                    "id": "demo_call_1",
                    "type": "tool_call",
                }],
            )
        ]
    }

runtime_builder = StateGraph(RuntimeState)
runtime_builder.add_node("create_call", create_demo_tool_call)
runtime_builder.add_node("tools", ToolNode([read_question_from_runtime]))
runtime_builder.add_edge(START, "create_call")
runtime_builder.add_edge("create_call", "tools")
runtime_builder.add_edge("tools", END)

runtime_graph = runtime_builder.compile()

runtime_result = runtime_graph.invoke({
    "question": "What is LangGraph?",
    "messages": [],
})

print("Tool result:", runtime_result["messages"][-1].content) -->

## 8. `Tool(...)` constructor"

In [32]:
from langchain_core.tools import Tool

In [33]:
def simple_search_function(query: str) -> str:
    return f"Searching for {query}"

In [35]:
simple_search_tool = Tool(
    name="simple_search",
    func=simple_search_function,
    description="Search for information.",
)

In [36]:
print("Name:", simple_search_tool.name)
print("Execution result:", simple_search_tool.invoke("LangGraph"))

Name: simple_search
Execution result: Searching for LangGraph


In [37]:
def search_from_function(query: str) -> str:
    return f"Result for {query}"

In [38]:
Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

Tool(name='search_from_function', description='Search information.', func=<function search_from_function at 0x000001269F8498A0>)

In [39]:
from_function_tool = Tool.from_function(
    func=search_from_function,
    name="search_from_function",
    description="Search information.",
)

In [40]:
print("Execution result:", from_function_tool.invoke("Agentic AI"))

Execution result: Result for Agentic AI


## 10. `StructuredTool.from_function()`

In [41]:
from langchain_core.tools import StructuredTool

def calculate_tax_test(income: float, tax_rate: float) -> float:
    return income * tax_rate

tax_tool_test = StructuredTool.from_function(
    func=calculate_tax_test,
    name="calculate_tax",
    description="Calculate tax from income and tax rate.",
)

print("Args:", tax_tool_test.args)
print("Execution result:", tax_tool_test.invoke({
    "income": 100000,
    "tax_rate": 0.20,
}))

Args: {'income': {'title': 'Income', 'type': 'number'}, 'tax_rate': {'title': 'Tax Rate', 'type': 'number'}}
Execution result: 20000.0


In [43]:
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")
class MultiplyInputTest2(BaseModel):
    a: int = Field(description="First number")
    b: int = Field(description="Second number")

def direct_multiply(a: int, b: int) -> int:
    return a * b

direct_structured_tool = StructuredTool(
    name="direct_multiply",
    description="Multiply two numbers.",
    func=direct_multiply,
    args_schema=MultiplyInputTest2,
)

print("Execution result:", direct_structured_tool.invoke({
    "a": 12,
    "b": 4,
}))

Execution result: 48


## 12. Subclass `BaseTool`

In [44]:
from typing import Type
from langchain_core.tools import BaseTool

In [45]:
class SearchInputTest(BaseModel):
    query: str = Field(description="Search query")

In [46]:
class MySearchToolTest(BaseTool):
    name: str = "my_search"
    description: str = "Search my custom database."
    args_schema: Type[BaseModel] = SearchInputTest

    def _run(self, query: str) -> str:
        return f"Custom database result for: {query}"

In [47]:
custom_base_tool = MySearchToolTest()

In [48]:
print("Execution result:", custom_base_tool.invoke({
    "query": "LangGraph state management"
}))

Execution result: Custom database result for: LangGraph state management


In [ ]:
1 class and react flow
2. memory
3. 
4.

practical:
1. agentic rag
2. multiagenticflow